# Resource Desert Analysis & Optimization
## UNF CodeforAwhile Datathon 2025 — Problem Case 1

This notebook orchestrates the full five-stage pipeline:

| Stage | Module | Purpose |
|---|---|---|
| 1 | `ingestion` | Load all 9 raw CSV datasets |
| 2 | `cleaning` | Standardise ZIPs, fix dtypes, clip rates |
| 3 | `features` | Merge, compute Desert Score |
| 4 | `models` | Gap-closure simulation & intervention ranking |
| 5 | `visualization` | Charts, choropleth map |

**Key questions answered:**
1. Which Jacksonville ZIP codes are the most underserved resource deserts?
2. How do resource deserts correlate with health outcomes?
3. Which single intervention would deliver the greatest improvement?

In [ ]:
import warnings
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless backend for reproducible output
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')

# Ensure src/ is importable from the notebook
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src import config as cfg
from src.ingestion import load_raw_datasets
from src.cleaning import clean_datasets
from src.features import (
    merge_datasets,
    compute_desert_score,
    compute_health_outcome_correlation,
    filter_by_service_category,
)
from src.models import run_gap_closure_simulation, rank_interventions
from src.visualization import (
    plot_desert_scores_bar_chart,
    create_choropleth_map,
    plot_preventative_vs_outcome,
    plot_intervention_impact_heatmap,
    plot_category_view,
)

print('Environment ready.')

---
## Stage 1 — Data Ingestion

Load all 9 raw CSV/XLSX source files from `data/raw/`.  No transformations are applied here.

In [ ]:
raw = load_raw_datasets()

print(f"Loaded {len(raw)} datasets:")
for key, df in raw.items():
    print(f"  {key:30s}  {df.shape[0]:>5} rows × {df.shape[1]:>3} cols")

---
## Stage 2 — Data Cleaning

- Standardise the ZIP column to `zip_code` (str, 5-char, zero-padded)
- Fix column dtypes
- Clip rate/proportion columns to `[0.0, 1.0]`
- Drop duplicates; log before/after row counts

In [ ]:
cleaned = clean_datasets(raw)

print("Post-cleaning shapes:")
for key, df in cleaned.items():
    print(f"  {key:30s}  {df.shape[0]:>5} rows × {df.shape[1]:>3} cols")

---
## Stage 3 — Feature Engineering

### 3a. Merge datasets

Outer-join all 9 cleaned datasets on `zip_code`, filter to Duval County ZIPs, impute null supply metrics with column medians.

In [ ]:
merged = merge_datasets(cleaned)
print(f"Merged dataset: {merged.shape[0]} ZIP codes × {merged.shape[1]} columns")
merged.head(3)

### 3b. Compute Desert Score

The **Resource Desert Score** (0–100, higher = more deprived) is computed as:

```
demand_factor  = normalise(poverty_rate × disease_burden_composite)
supply_gap_*   = normalised deprivation per resource type
desert_score   = normalise(Σ supply_gap_i × demand_factor) × 100
```

Ties are broken by `poverty_rate` descending.

In [ ]:
desert_scores = compute_desert_score(merged)
print(f"Desert scores computed for {len(desert_scores)} ZIP codes.")
print(f"Score range: [{desert_scores[cfg.COL_DESERT_SCORE].min():.1f}, "
      f"{desert_scores[cfg.COL_DESERT_SCORE].max():.1f}]")

### Top-10 Most Underserved ZIP Codes

In [ ]:
top10 = (
    desert_scores
    .nsmallest(10, cfg.COL_DESERT_RANK)
    [[cfg.COL_ZIP, cfg.COL_DESERT_RANK, cfg.COL_DESERT_SCORE,
      cfg.COL_TOP_GAP_CATEGORY, cfg.COL_TOTAL_POPULATION,
      cfg.COL_POVERTY_RATE]]
    .rename(columns={
        cfg.COL_ZIP: 'ZIP',
        cfg.COL_DESERT_RANK: 'Rank',
        cfg.COL_DESERT_SCORE: 'Desert Score',
        cfg.COL_TOP_GAP_CATEGORY: 'Top Gap',
        cfg.COL_TOTAL_POPULATION: 'Population',
        cfg.COL_POVERTY_RATE: 'Poverty Rate',
    })
    .set_index('Rank')
)
top10.style.format({'Desert Score': '{:.1f}', 'Poverty Rate': '{:.1%}'})

### Bar Chart — Top-10 Desert Scores

In [ ]:
plot_desert_scores_bar_chart(desert_scores)
print(f"Saved → {cfg.FIGURE_BAR_CHART}")

### Choropleth Map

The interactive map is saved as a standalone HTML file.  Open it in a browser to explore ZIP-level scores.

> **Note**: Requires `data/raw/jacksonville_zctas.geojson`.  The cell is skipped gracefully if the file is absent.

In [ ]:
if cfg.GEOJSON_PATH.exists():
    create_choropleth_map(desert_scores)
    print(f"Map saved → {cfg.CHOROPLETH_MAP_PATH}")
    from IPython.display import IFrame
    display(IFrame(str(cfg.CHOROPLETH_MAP_PATH), width=900, height=500))
else:
    print(f"GeoJSON not found at {cfg.GEOJSON_PATH} — skipping choropleth.")

---
## Stage 4 — Health Outcome Correlation (US2)

Quantify the Pearson correlation between each preventative asset metric and health outcome metric.

In [ ]:
corr_df = compute_health_outcome_correlation(merged)
print("Pearson correlation — Preventative Assets vs Health Outcomes:")

# Pivot to wide matrix (assets as rows, outcomes as columns)
corr_matrix = corr_df.pivot(index="asset", columns="outcome", values="pearson_r")
corr_matrix.index.name = "Asset"
corr_matrix.columns.name = "Outcome"
corr_matrix.style.background_gradient(cmap="RdYlGn", vmin=-1, vmax=1).format("{:.3f}")

In [ ]:
# Plot the strongest association
plot_preventative_vs_outcome(
    merged,
    asset_col=cfg.COL_PRIMARY_CARE_RATIO,
    outcome_col=cfg.COL_POOR_MENTAL_HEALTH,
)
print(f"Scatter plot saved → {cfg.FIGURE_HEALTH_SCATTER}")

---
## Stage 5 — Gap-Closure Simulation (US3)

For each of the **top-5 most underserved ZIPs** × **4 resource types**, simulate setting the supply metric to the 25th-percentile target and compute the Desert Score improvement.

In [ ]:
interventions = run_gap_closure_simulation(desert_scores, merged, top_n=5)
ranked = rank_interventions(interventions)

best = ranked[ranked['is_highest_impact']].iloc[0]
print("=" * 60)
print("HIGHEST-IMPACT INTERVENTION")
print("=" * 60)
print(f"  ZIP Code        : {best['zip_code']}")
print(f"  Resource Type   : {best['resource_type']}")
print(f"  Current Score   : {best['current_desert_score']:.1f}")
print(f"  Simulated Score : {best['simulated_desert_score']:.1f}")
print(f"  Improvement     : {best['score_improvement']:.1f} pts ({best['pct_improvement']:.1f}%)")
print(f"  Population      : {best['population_impacted']:,}")
print(f"\nFull recommendations → {cfg.INTERVENTION_RECOMMENDATIONS_PATH}")

In [ ]:
plot_intervention_impact_heatmap(ranked)
print(f"Heatmap saved → {cfg.FIGURE_INTERVENTION_HEATMAP}")

### Intervention Recommendations Summary

In [ ]:
ranked[['zip_code', 'resource_type', 'score_improvement',
        'pct_improvement', 'population_impacted', 'is_highest_impact']].style.apply(
    lambda row: ['background-color: #ffe066' if row['is_highest_impact'] else '' for _ in row],
    axis=1
).format({'score_improvement': '{:.2f}', 'pct_improvement': '{:.1f}%'})

---
## Stage 6 — Per-Category Drill-Down (US4)

Any service category can be isolated and visualised independently.

In [ ]:
for category in ['healthcare', 'food_access', 'parks', 'insurance']:
    cat_df = filter_by_service_category(desert_scores, category)
    plot_category_view(cat_df, category=category)
    out = cfg.REPORTS_FIGURES_DIR / f'category_{category}_view.png'
    print(f"  {category:15s} — {len(cat_df)} ZIPs — saved → {out}")

---
## Summary

| Output | Location |
|---|---|
| Desert scores CSV | `reports/outputs/desert_scores.csv` |
| Intervention recommendations JSON | `reports/outputs/intervention_recommendations.json` |
| Choropleth map HTML | `reports/outputs/resource_desert_map.html` |
| Bar chart | `reports/figures/desert_scores_bar_chart.png` |
| Health scatter | `reports/figures/preventative_asset_vs_health_outcome.png` |
| Intervention heatmap | `reports/figures/intervention_impact_heatmap.png` |
| Category views (×4) | `reports/figures/category_{name}_view.png` |